# Advanced FileSet Features

This notebook demonstrates advanced FileSet capabilities:
- **Metadata filtering** — restrict which documents are used as seeds
- **Query-based seeds** — use semantic queries instead of chunking
- **RAG context generation** — retrieve supporting context with temporal constraints
- **RAG labeling** — resolve questions by searching the FileSet
- **Full combined pipeline** — context + labeling in one run

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [ ]:
%pip install lightningrod-ai python-dotenv pandas -q

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [ ]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [ ]:
fileset_id = "PASTE_YOUR_FILESET_ID_HERE"

In [ ]:
import pandas as pd
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    FileSetQuerySeedGenerator,
    FileSetContextGenerator,
    FileSetRAGLabeler,
    QuestionGenerator,
    BinaryAnswerType,
    TemporalConstraint,
)

answer_type = BinaryAnswerType()

## Metadata Filtering

Use `metadata_filters` on the seed generator to restrict which files become seeds. Here we generate questions only from **APEX** documents.

In [ ]:
pipeline_filtered = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='APEX'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the specific financial metrics and business events in these quarterly reports.",
        questions_per_seed=2,
    ),
)

dataset_filtered = lr.transforms.run(
    pipeline_filtered,
    max_questions=6,
    name="FileSet - APEX Only (Metadata Filter)",
)
print(f"Dataset: {dataset_filtered.id}")
print(f"Rows: {dataset_filtered.num_rows}")

In [ ]:
samples_filtered = dataset_filtered.download()
for i, s in enumerate(samples_filtered[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"Seed (first 120 chars): {s.seed.seed_text[:120]}...")
    print(f"Question: {s.question.question_text}")
    print()

## Query-Based Seeds

`FileSetQuerySeedGenerator` runs semantic queries against the FileSet instead of chunking all documents. This is useful when you want seeds focused on specific topics. Here we query **VGI** documents only.

In [ ]:
pipeline_query = QuestionPipeline(
    seed_generator=FileSetQuerySeedGenerator(
        file_set_id=fileset_id,
        prompts=[
            "What is the current status and growth trajectory of the Robotics-as-a-Service (RaaS) segment?",
            "What restructuring actions has the company taken and what are the expected cost savings?",
            "What are the company's revenue guidance figures for the next quarter and full year?",
        ],
        metadata_filters=["ticker='VGI'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the specific facts in the retrieved content.",
        questions_per_seed=2,
    ),
)

dataset_query = lr.transforms.run(
    pipeline_query,
    max_questions=6,
    name="FileSet - Query Seeds (VGI)",
)
print(f"Dataset: {dataset_query.id}")
print(f"Rows: {dataset_query.num_rows}")

In [ ]:
samples_query = dataset_query.download()
rows = dataset_query.flattened(answer_type)
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "is_valid"]
df[[c for c in cols if c in df.columns]]

## RAG Context Generation

`FileSetContextGenerator` retrieves supporting context from the FileSet for each generated question.

- **`metadata_filter_keys=["ticker"]`** — only retrieve context from the same company
- **`temporal_constraint=BEFORE`** — only retrieve context from documents dated before the seed, preventing lookahead bias

In [ ]:
pipeline_context = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the financial performance and business events in these investor reports.",
        questions_per_seed=1,
    ),
    context_generators=[
        FileSetContextGenerator(
            file_set_id=fileset_id,
            metadata_filter_keys=["ticker"],
            temporal_constraint=TemporalConstraint.BEFORE,
        ),
    ],
)

dataset_context = lr.transforms.run(
    pipeline_context,
    max_questions=5,
    name="FileSet - Context Generation (BEFORE)",
)
print(f"Dataset: {dataset_context.id}")
print(f"Rows: {dataset_context.num_rows}")

In [ ]:
samples_context = dataset_context.download()
for i, s in enumerate(samples_context[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"Question: {s.question.question_text}")
    if s.context:
        for j, ctx in enumerate(s.context):
            rendered = getattr(ctx, 'rendered_context', str(ctx))
            print(f"  Context {j+1} (first 200 chars): {str(rendered)[:200]}...")
    else:
        print("  Context: None")
    print()

## RAG Labeling

`FileSetRAGLabeler` resolves questions by searching the FileSet for answers.

- **`temporal_constraint=AFTER`** — only search documents dated after the seed, so forward-looking questions are resolved by later reports
- **`confidence_threshold=0.7`** — only label questions where the labeler is at least 70% confident

In [ ]:
pipeline_labeler = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='VGI'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements, guidance, and planned initiatives "
            "mentioned in these quarterly reports. Focus on questions whose answers would be found in "
            "subsequent quarterly reports."
        ),
        questions_per_seed=2,
    ),
    labeler=FileSetRAGLabeler(
        file_set_id=fileset_id,
        metadata_filter_keys=["ticker"],
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_labeler = lr.transforms.run(
    pipeline_labeler,
    max_questions=6,
    name="FileSet - RAG Labeler (VGI, AFTER)",
)
print(f"Dataset: {dataset_labeler.id}")
print(f"Rows: {dataset_labeler.num_rows}")

In [ ]:
samples_labeler = dataset_labeler.download()
rows = dataset_labeler.flattened(answer_type)
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "reasoning"]
df[[c for c in cols if c in df.columns]]

## Full Pipeline — Context + Labeling

Combine context generation and labeling in a single pipeline:
- **Context** (`BEFORE`) — retrieve earlier reports as supporting context
- **Labeler** (`AFTER`) — resolve forward-looking questions using later reports

In [ ]:
pipeline_full = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements and guidance in these investor reports. "
            "Focus on questions that can be verified by looking at later quarterly reports for the same company."
        ),
        questions_per_seed=1,
    ),
    context_generators=[
        FileSetContextGenerator(
            file_set_id=fileset_id,
            metadata_filter_keys=["ticker"],
            temporal_constraint=TemporalConstraint.BEFORE,
        ),
    ],
    labeler=FileSetRAGLabeler(
        file_set_id=fileset_id,
        metadata_filter_keys=["ticker"],
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_full = lr.transforms.run(
    pipeline_full,
    max_questions=8,
    name="FileSet - Full Pipeline (Context + Labeler)",
)
print(f"Dataset: {dataset_full.id}")
print(f"Rows: {dataset_full.num_rows}")

In [ ]:
samples_full = dataset_full.download()
for i, s in enumerate(samples_full[:4]):
    print(f"--- Sample {i+1} ---")
    print(f"Question: {s.question.question_text}")
    if s.context:
        for j, ctx in enumerate(s.context):
            rendered = getattr(ctx, 'rendered_context', str(ctx))
            print(f"  Context {j+1} (first 150 chars): {str(rendered)[:150]}...")
    else:
        print("  Context: None")
    if s.label:
        print(f"  Label: {s.label.label} (confidence: {s.label.label_confidence})")
        print(f"  Reasoning: {s.label.reasoning}")
    else:
        print("  Label: None")
    print()

In [ ]:
rows = dataset_full.flattened(answer_type)
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "reasoning", "is_valid"]
df[[c for c in cols if c in df.columns]]